In [1]:
import re
import xml.etree.ElementTree as ET
from datetime import datetime
from urllib.parse import quote

import pandas as pd
import requests

BUCKET_URL = "https://s3-eu-west-1.amazonaws.com/cycling.data.tfl.gov.uk/"
FILE_URL = "https://cycling.data.tfl.gov.uk/"
YEAR = 2022


def list_keys(prefix="usage-stats/"):
    """List every file key in the bucket under the given prefix (handles paging)."""
    keys, marker = [], ""
    while True:
        r = requests.get(BUCKET_URL, params={"prefix": prefix, "marker": marker}, timeout=30)
        r.raise_for_status()
        root = ET.fromstring(r.content)
        ns = {"s3": root.tag.split("}")[0].strip("{")}
        page = [c.find("s3:Key", ns).text for c in root.findall("s3:Contents", ns)]
        keys += page
        if root.find("s3:IsTruncated", ns).text != "true" or not page:
            break
        marker = page[-1]
    return keys


DATE_RANGE = re.compile(r"(\d{1,2}\s?[A-Za-z]{3}\s?\d{4})\s?-\s?(\d{1,2}\s?[A-Za-z]{3}\s?\d{4})")


def file_overlaps_year(key, year):
    """True if the date range in the filename touches the given year."""
    m = DATE_RANGE.search(key)
    if not m:
        return False
    try:
        start, end = (datetime.strptime(d.replace(" ", ""), "%d%b%Y") for d in m.groups())
    except ValueError:
        return False
    return start.year <= year <= end.year


import io

# --- Step 1: did the listing work? ---
all_keys = list_keys()
print(f"Keys in bucket under usage-stats/: {len(all_keys)}")
print("Sample:", all_keys[:3])

csv_keys = [k for k in all_keys if k.lower().endswith(".csv") and "JourneyDataExtract" in k]
print(f"JourneyDataExtract CSVs: {len(csv_keys)}")

keys = [k for k in csv_keys if file_overlaps_year(k, YEAR)]
print(f"Files overlapping {YEAR}: {len(keys)}")
for k in keys[:3]:
    print("  ", k)

# --- Step 2: did the downloads work? ---
import time

RENAME = {
    # old layout                  # new layout
    "rental id": "rental_id",     "number": "rental_id",
    "bike id": "bike_id",         "bike number": "bike_id",
    "start date": "start_date",
    "end date": "end_date",
    "startstation id": "start_station_id",   "start station number": "start_station_id",
    "startstation name": "start_station",    "start station": "start_station",
    "endstation id": "end_station_id",       "end station number": "end_station_id",
    "endstation name": "end_station",        "end station": "end_station",
    "bike model": "bike_model",
    "duration": "duration_s",                # old layout: seconds
    "total duration (ms)": "duration_ms",    # new layout: milliseconds
    "total duration": "duration_text",       # new layout: text like "17m 5s"
}

def parse_dates(s):
    """Parse a mix of ISO (2022-03-23 14:05) and UK (23/03/2022 14:05) date strings."""
    s = s.astype("string").str.strip()
    iso = s.str.match(r"^\d{4}-\d{2}-\d{2}", na=False)
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    out[iso] = pd.to_datetime(s[iso], format="ISO8601", errors="coerce")
    out[~iso] = pd.to_datetime(s[~iso], format="mixed", dayfirst=True, errors="coerce")
    return out

def download(url, attempts=5):
    """GET with retries, so temporary network errors don't lose a file."""
    for attempt in range(1, attempts + 1):
        try:
            r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=120)
            r.raise_for_status()
            return r.content
        except Exception as e:
            if attempt == attempts:
                raise
            wait = 5 * attempt
            print(f"(retry {attempt}/{attempts - 1} after {type(e).__name__}, waiting {wait}s)", end=" ", flush=True)
            time.sleep(wait)


def load_file(key):
    url = FILE_URL + quote(key)
    df = pd.read_csv(
        io.BytesIO(download(url)),
        dtype={
            "Start station number": "string", "End station number": "string",
            "StartStation Id": "string", "EndStation Id": "string",
        },
    )
    df.columns = df.columns.str.strip().str.lower()
    df = df.rename(columns=RENAME)

    for c in ["start_station_id", "end_station_id"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    if "duration_ms" in df.columns:
        df["duration_s"] = df["duration_ms"] / 1000
    df = df.drop(columns=["duration_ms", "duration_text"], errors="ignore")

    for c in ["start_date", "end_date"]:
        raw = df[c]
        df[c] = parse_dates(raw)
        bad = df[c].isna() & raw.notna()
        if bad.any():
            print(f"\n   {bad.sum()} unparseable in {c}, e.g. {raw[bad].unique()[:3].tolist()}")

    df["source_file"] = key.split("/")[-1]
    return df


frames, failed = [], []
for i, key in enumerate(keys, 1):
    print(f"[{i}/{len(keys)}] {key} ...", end=" ", flush=True)
    try:
        df = load_file(key)
    except Exception as e:
        print(f"FAILED: {type(e).__name__}: {e}")
        failed.append(key)
        continue
    print(f"{len(df)} rows")
    frames.append(df)

del df
print(f"Loaded {len(frames)} of {len(keys)} files. Failed: {failed}")

rentals = pd.concat(frames, ignore_index=True)
rentals = rentals[rentals["start_date"].dt.year == YEAR].reset_index(drop=True)
print(rentals.shape)
print(rentals["source_file"].value_counts().sort_index())

Keys in bucket under usage-stats/: 482
Sample: ['usage-stats/', 'usage-stats/01aJourneyDataExtract10Jan16-23Jan16.csv', 'usage-stats/01b Journey Data Extract 24Jan16-06Feb16.csv']
JourneyDataExtract CSVs: 461
Files overlapping 2022: 53
   usage-stats/298JourneyDataExtract29Dec2021-04Jan2022.csv
   usage-stats/299JourneyDataExtract05Jan2022-11Jan2022.csv
   usage-stats/300JourneyDataExtract12Jan2022-18Jan2022.csv
[1/53] usage-stats/298JourneyDataExtract29Dec2021-04Jan2022.csv ... 128938 rows
[2/53] usage-stats/299JourneyDataExtract05Jan2022-11Jan2022.csv ... 134383 rows
[3/53] usage-stats/300JourneyDataExtract12Jan2022-18Jan2022.csv ... 180204 rows
[4/53] usage-stats/301JourneyDataExtract19Jan2022-25Jan2022.csv ... 182437 rows
[5/53] usage-stats/302JourneyDataExtract26Jan2022-01Feb2022.csv ... 203961 rows
[6/53] usage-stats/303JourneyDataExtract02Feb2022-08Feb2022.csv ... 198653 rows
[7/53] usage-stats/304JourneyDataExtract09Feb2022-15Feb2022.csv ... 181350 rows
[8/53] usage-stats/305Jo

In [4]:
rentals.info()
rentals.to_csv("rentals2022.csv", index=False)

<class 'pandas.DataFrame'>
RangeIndex: 11419294 entries, 0 to 11419293
Data columns (total 11 columns):
 #   Column            Dtype         
---  ------            -----         
 0   rental_id         int64         
 1   duration_s        float64       
 2   bike_id           int64         
 3   end_date          datetime64[ns]
 4   end_station_id    Int64         
 5   end_station       str           
 6   start_date        datetime64[ns]
 7   start_station_id  Int64         
 8   start_station     str           
 9   source_file       str           
 10  bike_model        str           
dtypes: Int64(2), datetime64[ns](2), float64(1), int64(2), str(4)
memory usage: 2.0 GB


In [5]:
print(rentals.head())

   rental_id  duration_s  bike_id            end_date  end_station_id  \
0  115967515      1260.0    15338 2022-01-01 23:13:00             310   
1  116017034       720.0    19861 2022-01-04 19:08:00              11   
2  116016563       480.0    19861 2022-01-04 18:46:00             804   
3  116014412      1260.0    17235 2022-01-04 17:45:00              14   
4  116013350       480.0    13790 2022-01-04 16:50:00             252   

                      end_station          start_date  start_station_id  \
0     Black Prince Road, Vauxhall 2022-01-01 22:52:00               529   
1    Brunswick Square, Bloomsbury 2022-01-04 18:56:00               804   
2        Good's Way, King's Cross 2022-01-04 18:38:00                57   
3  Belgrove Street , King's Cross 2022-01-04 17:24:00               297   
4     Jubilee Gardens, South Bank 2022-01-04 16:42:00               310   

                         start_station  \
0                Manresa Road, Chelsea   
1             Good's Way, 